In [1]:
#0) Setup & Imports

In [2]:
# =========================
# ETL: BTC + Sentiment + Hash-rate → Hourly Master (+ Daily EOD 23:59)
# Python 3.12 / JupyterLab
# =========================
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# ---------- Paths ----------
PATH_BTC    = "data/btc1h_usdt.csv"
PATH_SENT   = "data/bitcoin_sentiments_21_24.csv"   # optional
PATH_HASH   = "data/hash_rate.csv"                  # optional

OUT_CSV_H   = "data/out/btc_master_hourly_2017_2025.csv"
OUT_PQ_H    = "data/out/btc_master_hourly_2017_2025.parquet"

OUT_CSV_D   = "data/out/btc_master_daily_eod_2017_2025.csv"
OUT_PQ_D    = "data/out/btc_master_daily_eod_2017_2025.parquet"

# Which timezone defines the "day" for EOD anchoring (23:59)
EOD_TZ = "UTC"  # set to "Europe/Lisbon" if you want local-day boundaries

# Daily features lag to ensure strict causality
SAFE_DAILY_LAG_DAYS = 1  # keep 1 unless your daily sources are already "as-of previous close"

# ---------- Helpers ----------
def _find_datetime_col(df: pd.DataFrame):
    for c in ["open_time", "timestamp", "date", "datetime", "time", "Date"]:
        if c in df.columns:
            return c
    return None

def _coerce_numeric(df: pd.DataFrame, skip_cols=None) -> pd.DataFrame:
    skip_cols = set(skip_cols or [])
    out = df.copy()
    for c in out.columns:
        if c in skip_cols:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def load_btc(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"BTC file not found: {path}")
    df = pd.read_csv(path)
    dt = _find_datetime_col(df)
    if dt is None:
        raise ValueError("No datetime column found in BTC CSV (expected open_time/timestamp/date/datetime/time).")
    df[dt] = pd.to_datetime(df[dt], utc=True)
    df = df.sort_values(dt).set_index(dt)

    # Ensure 'close' exists and numeric
    if "close" not in df.columns:
        cands = [c for c in df.columns if c.lower() == "close"]
        if not cands:
            raise ValueError("BTC CSV must contain a 'close' (price) column.")
        df["close"] = df[cands[0]]
    df["close"] = pd.to_numeric(df["close"], errors="coerce")

    # Authoritative hourly grid; explicit missing rows (NaNs)
    df = df[~df.index.duplicated(keep="first")].asfreq("1H")
    return df

def load_sentiment_daily(path: str):
    if not os.path.exists(path):
        return None
    s = pd.read_csv(path)
    dt = _find_datetime_col(s)
    if dt is None:
        raise ValueError("No datetime column found in sentiment CSV.")
    s[dt] = pd.to_datetime(s[dt], utc=True)
    s = s.sort_values(dt).set_index(dt)
    s = s.rename(columns={c: (c if str(c).lower().startswith("sent_") else f"sent_{c}") for c in s.columns})
    s_num = _coerce_numeric(s)
    return s_num.resample("1D").mean()  # daily cadence before causality fix

def load_hash_daily(path: str):
    if not os.path.exists(path):
        return None
    h = pd.read_csv(path)
    dt = _find_datetime_col(h)
    if dt is None:
        raise ValueError("No datetime column found in hash-rate CSV.")
    h[dt] = pd.to_datetime(h[dt], utc=True)
    h = h.sort_values(dt).set_index(dt)
    h_num = _coerce_numeric(h)
    return h_num.resample("1D").mean()  # daily cadence before causality fix

def causalize_daily(daily: pd.DataFrame, source_name: str, lag_days: int = SAFE_DAILY_LAG_DAYS) -> pd.DataFrame:
    """
    Make a daily frame strictly causal:
    - Keep provenance column '{source}_asof_date' (the source day the value belongs to, before lagging).
    - Shift by lag_days so Day D values are only used starting Day D+lag_days.
    """
    if daily is None or daily.empty:
        return daily
    out = daily.copy()
    out[f"{source_name}_asof_date"] = out.index.normalize()  # midnight UTC for that source day
    if lag_days:
        out = out.shift(lag_days)
    return out

def assert_no_future_leakage(master: pd.DataFrame, sources=("sent", "hash")):
    """
    Verify that for every hourly timestamp T (UTC), the '{src}_asof_date' used is <= T-1 day (UTC calendar).
    Raises AssertionError if any violation is found.
    """
    if master is None or master.empty:
        return
    ts_day = pd.Series(master.index.normalize(), index=master.index)
    prev_day = ts_day - pd.Timedelta(days=1)

    problems = []
    for src in sources:
        col = f"{src}_asof_date"
        if col in master.columns:
            asof = pd.to_datetime(master[col]).dt.normalize()
            bad = asof > prev_day
            if bad.any():
                n = int(bad.sum())
                examples = master.index[bad][:5].astype(str).tolist()
                problems.append(f"{src}: {n} rows (e.g., {examples})")
    if problems:
        raise AssertionError("Causality check failed (future leakage in daily features): " + " | ".join(problems))

def daily_close_2359(hourly: pd.DataFrame, tz: str = "UTC") -> pd.DataFrame:
    """
    Take last hourly bar of each calendar day (in tz), anchor its timestamp to 23:59,
    and add a 'close_date' column (date only).
    """
    if hourly.empty:
        return hourly.copy()
    h = hourly.copy()
    h.index = h.index.tz_convert(tz)
    d = h.resample("1D").last()           # last bar per local calendar day (handles DST)
    d.index = d.index + pd.Timedelta(hours=23, minutes=59)  # move to 23:59 of same day
    d["close_date"] = d.index.date

    if "close" in d.columns:
        d["log_close"] = np.log(d["close"])
        d["ret_d"]     = d["log_close"].diff()
        d["y_ret_d1"]  = d["ret_d"].shift(-1)
        d["y_dir_d1"]  = np.sign(d["y_ret_d1"]).astype("float32")
    return d

def integrity_report(frame: pd.DataFrame, index_freq="1H", max_list=10):
    if frame.empty:
        print("Dataset is empty.")
        return
    idx_min, idx_max = frame.index.min(), frame.index.max()
    full_range = pd.date_range(idx_min, idx_max, freq=index_freq)
    missing_bars = len(full_range.difference(frame.index))
    dups = frame.index.duplicated().sum()
    n_rows, n_cols = frame.shape

    nan_counts = frame.isna().sum().sort_values(ascending=False)
    top_nan = nan_counts[nan_counts > 0].head(max_list)

    print("=== Integrity Report ===")
    print("• Index range:", idx_min, "→", idx_max)
    print(f"• Rows: {n_rows} | Cols: {n_cols}")
    print("• Duplicate timestamps:", dups)
    print(f"• Missing bars ({index_freq}):", missing_bars)
    if len(top_nan) > 0:
        print("• Top NaN columns:")
        print(top_nan)

# ---------- ETL Pipeline ----------
# 1) Load sources
btc = load_btc(PATH_BTC)
sent_daily = load_sentiment_daily(PATH_SENT)
hash_daily = load_hash_daily(PATH_HASH)

print(f"Loaded BTC hourly: {btc.shape}")
print(f"Loaded Sentiment daily: {None if sent_daily is None else sent_daily.shape}")
print(f"Loaded Hash-rate daily: {None if hash_daily is None else hash_daily.shape}")

# 2) Start from BTC hourly grid
master = btc.copy()

# 3) Sentiment: daily → causalize (lag) → ffill to hourly on BTC grid
if sent_daily is not None and not sent_daily.empty:
    sent_daily = causalize_daily(sent_daily, source_name="sent", lag_days=SAFE_DAILY_LAG_DAYS)
    sent_idx = pd.date_range(
        start=sent_daily.index.min(),
        end=max(sent_daily.index.max(), master.index.max()),
        freq="1D",
        tz=sent_daily.index.tz,
    )
    sent_daily = sent_daily.reindex(sent_idx).ffill()
    sent_hourly = sent_daily.reindex(master.index, method="ffill")
    master = master.join(sent_hourly, how="left")

# 4) Hash-rate: daily → causalize (lag) → ffill to hourly on BTC grid
if hash_daily is not None and not hash_daily.empty:
    hash_daily = causalize_daily(hash_daily, source_name="hash", lag_days=SAFE_DAILY_LAG_DAYS)
    hash_idx = pd.date_range(
        start=hash_daily.index.min(),
        end=max(hash_daily.index.max(), master.index.max()),
        freq="1D",
        tz=hash_daily.index.tz,
    )
    hash_daily = hash_daily.reindex(hash_idx).ffill()
    hash_hourly = hash_daily.reindex(master.index, method="ffill")
    master = master.join(hash_hourly, how="left")

# 5) Hourly targets (leakage-safe)
master["log_close"] = np.log(master["close"])
master["ret_t"]     = master["log_close"].diff()
master["y_ret_t1"]  = master["ret_t"].shift(-1)
master["y_dir_t1"]  = np.sign(master["y_ret_t1"]).astype("float32")
master = master.iloc[:-1]  # drop final hour where y_ret_t1 is NaN

# 5b) Safety net: prove we never used same-day info for daily features
assert_no_future_leakage(master, sources=("sent", "hash"))

# 5c) Daily EOD view normalized to 23:59 (reporting-friendly)
daily_eod = daily_close_2359(master, tz=EOD_TZ)
daily_eod = daily_eod.iloc[:-1]  # drop last day if daily target used

# 6) Integrity & samples
print("\n-- HOURLY --")
integrity_report(master, index_freq="1H")
print("\nHead:")
print(master.head(5))
print("\nTail:")
print(master.tail(5))

print("\n-- DAILY (EOD 23:59, tz=%s) --" % EOD_TZ)
integrity_report(daily_eod, index_freq="1D")
print("\nHead:")
print(daily_eod.head(3)[["close", "close_date"]])
print("\nTail:")
print(daily_eod.tail(3)[["close", "close_date"]])

# 7) Save outputs
os.makedirs(os.path.dirname(OUT_CSV_H), exist_ok=True)
master.to_parquet(OUT_PQ_H)
master.to_csv(OUT_CSV_H)
daily_eod.to_parquet(OUT_PQ_D)
daily_eod.to_csv(OUT_CSV_D)
print(f"\nSaved datasets:\n - {OUT_PQ_H}\n - {OUT_CSV_H}\n - {OUT_PQ_D}\n - {OUT_CSV_D}")


Loaded BTC hourly: (64862, 111)
Loaded Sentiment daily: (1043, 2)
Loaded Hash-rate daily: (1, 2)

-- HOURLY --
=== Integrity Report ===
• Index range: 2017-08-17 04:00:00+00:00 → 2025-01-09 16:00:00+00:00
• Rows: 64861 | Cols: 121
• Duplicate timestamps: 0
• Missing bars (1H): 0
• Top NaN columns:
sent_Short Description      64861
date                        64861
hash-rate                   64861
hash_asof_date              64861
momentum_pvo                59401
momentum_pvo_signal         59401
momentum_pvo_hist           59401
sent_asof_date              37004
sent_Accurate Sentiments    37004
trend_psar_down             33553
dtype: int64

Head:
                           Unnamed: 0     open     high      low    close  \
open_time                                                                   
2017-08-17 04:00:00+00:00         0.0  4261.48  4313.62  4261.32  4308.83   
2017-08-17 05:00:00+00:00         1.0  4308.83  4328.69  4291.37  4315.32   
2017-08-17 06:00:00+00:00        